# Introduction

This notebook serves as a small exercise to practice using Scikit-learn. While I am not familiar with how this task was approached in the past, I have recently started learning Python and exploring small projects. From my experience so far, Scikit-learn appears to be a fundamental toolkit for machine learning tasks, making it a natural choice for this practice.

# Scenario

All the information you should know is [here](https://www.kaggle.com/competitions/data-science-london-scikit-learn).

# Data Exploration and Analysis

The dataset consists of preprocessed feature vectors, allowing us to proceed directly with model training and analysis without the need for additional data cleaning or feature engineering.

In [ ]:
import pandas as pd

# 没有表头，所以 header=None
X_train = pd.read_csv('/kaggle/input/data-science-london-scikit-learn/train.csv', header=None)
y_train = pd.read_csv('/kaggle/input/data-science-london-scikit-learn/trainLabels.csv', header=None).values.ravel()
X_test = pd.read_csv('/kaggle/input/data-science-london-scikit-learn/test.csv', header=None)
X_train


# Model Building

I did not attempt extensive model experimentation or hyperparameter tuning to achieve an optimal score. Instead, I focused on repeatedly using Scikit-learn throughout this exercise, as my main goal was to gain hands-on practice with the library.

In [ ]:
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, VotingClassifier
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
import lightgbm as lgb
import xgboost as xgb
from sklearn.metrics import accuracy_score
from sklearn.model_selection import StratifiedKFold

# 1. 读取数据
X_train = pd.read_csv('/kaggle/input/data-science-london-scikit-learn/train.csv', header=None)
y_train = pd.read_csv('/kaggle/input/data-science-london-scikit-learn/trainLabels.csv', header=None).values.ravel()
X_test = pd.read_csv('/kaggle/input/data-science-london-scikit-learn/test.csv', header=None)

# 2. 定义多个模型（默认参数）
model1 = LogisticRegression(max_iter=1000)
model2 = RandomForestClassifier()
model3 = GradientBoostingClassifier()
model4 = lgb.LGBMClassifier()
model5 = xgb.XGBClassifier(use_label_encoder=False, eval_metric='logloss')
model6 = MLPClassifier(max_iter=500)

# 3. Voting 集成器（Soft voting）
ensemble_model = VotingClassifier(
    estimators=[
        ('lr', model1),
        ('rf', model2),
        ('gb', model3),
        ('lgb', model4),
        ('xgb', model5),
        ('mlp', model6),
    ],
    voting='soft',  # 概率投票
    n_jobs=-1
)

# 4. 可选：交叉验证看效果
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores = []
for train_idx, val_idx in skf.split(X_train, y_train):
    X_tr, X_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
    y_tr, y_val = y_train[train_idx], y_train[val_idx]

    ensemble_model.fit(X_tr, y_tr)
    val_pred = ensemble_model.predict(X_val)
    acc = accuracy_score(y_val, val_pred)
    scores.append(acc)
print(f"Ensemble CV Accuracy: {sum(scores)/len(scores):.4f}")

# 5. 用全部数据训练最终模型
ensemble_model.fit(X_train, y_train)

# 6. 预测测试集
test_pred = ensemble_model.predict(X_test)

# 7. 写入提交文件
submission = pd.DataFrame(test_pred, columns=["Solution"])
submission.index += 1  # Id 从 1 开始
submission.index.name = "Id"
submission.to_csv("submission.csv")


# Conclusion

Each stage of learning comes with its own focus. For now, I consider myself at a beginner level, and I believe it's important to observe more, think more, and practice consistently. That said, the pace of development in data analysis and related fields is truly remarkable.